# Streaming - Producer de eventos simulados

Objetivo: simular a chegada contínua de novas medições do indicador de alfabetização
por município (um evento por vez), no formato que o Firehose vai receber e gravar
em `s3://.../bronze_stream/`.

In [1]:
import awswrangler as wr
import pandas as pd

BUCKET = "brazil-literacy-lakehouse-joaopaulo"

df_municipio = wr.s3.read_parquet(
    path=f"s3://{BUCKET}/bronze/municipio/",
    dataset=True,
)

df_municipio.shape

(23995, 15)

## Geração do evento

In [2]:
import uuid
import json
from datetime import datetime, timezone

def gerar_evento(linha):
    return {
        "evento_id": str(uuid.uuid4()),
        "tipo_evento": "nova_medicao_indicador",
        "timestamp_evento": datetime.now(timezone.utc).isoformat(),
        "ano": int(linha["ano"]),
        "id_municipio": str(linha["id_municipio"]),
        "serie": int(linha["serie"]) if pd.notna(linha["serie"]) else None,
        "rede": int(linha["rede"]) if pd.notna(linha["rede"]) else None,
        "taxa_alfabetizacao": float(linha["taxa_alfabetizacao"]) if pd.notna(linha["taxa_alfabetizacao"]) else None,
        "fonte": "simulacao_streaming",
    }

In [3]:
# Sanity check: gera o evento pra uma amostra pequena e confere o formato
amostra = df_municipio.sample(3, random_state=42)

for _, linha in amostra.iterrows():
    evento = gerar_evento(linha)
    print(json.dumps(evento, indent=2, ensure_ascii=False))

{
  "evento_id": "d875d4cf-a51e-480d-b312-d8bdde297b6f",
  "tipo_evento": "nova_medicao_indicador",
  "timestamp_evento": "2026-08-28T05:13:15.269733+00:00",
  "ano": 2024,
  "id_municipio": "3166956",
  "serie": 2,
  "rede": 3,
  "taxa_alfabetizacao": 95.99,
  "fonte": "simulacao_streaming"
}
{
  "evento_id": "a6512ab9-51d3-4043-adc8-80b086b14f4d",
  "tipo_evento": "nova_medicao_indicador",
  "timestamp_evento": "2026-08-28T05:13:15.270247+00:00",
  "ano": 2023,
  "id_municipio": "5219605",
  "serie": 2,
  "rede": 3,
  "taxa_alfabetizacao": 96.21,
  "fonte": "simulacao_streaming"
}
{
  "evento_id": "2776a3d4-e4b1-45c6-8ddd-2f05c0b6119a",
  "tipo_evento": "nova_medicao_indicador",
  "timestamp_evento": "2026-08-28T05:13:15.270443+00:00",
  "ano": 2023,
  "id_municipio": "3125952",
  "serie": 2,
  "rede": 5,
  "taxa_alfabetizacao": 66.12,
  "fonte": "simulacao_streaming"
}


## Envio para o Firehose (`stream-eventos-alfabetizacao`)

In [4]:
import boto3

REGION = "sa-east-1"
PREFIX = "bronze_stream/"
STREAM_NAME = "stream-eventos-alfabetizacao"

firehose = boto3.client("firehose", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)


def enviar_evento(evento: dict):
    """Envia 1 evento via put_record — útil pra testes rápidos."""
    return firehose.put_record(
        DeliveryStreamName=STREAM_NAME,
        Record={"Data": (json.dumps(evento) + "\n").encode("utf-8")}
    )

In [5]:
import time
import math

BATCH_SIZE = 500  # limite do Firehose por chamada de put_record_batch


def gerar_todos_eventos(df):
    """Gera a lista de eventos (dicts) a partir do DataFrame de municípios."""
    return [gerar_evento(linha) for _, linha in df.iterrows()]


def enviar_lote(records: list[dict], max_retries: int = 3):
    """Envia uma lista de eventos via put_record_batch, com retry pros que falharem."""
    entries = [
        {"Data": (json.dumps(evento) + "\n").encode("utf-8")}
        for evento in records
    ]

    tentativa = 0
    while entries and tentativa <= max_retries:
        response = firehose.put_record_batch(
            DeliveryStreamName=STREAM_NAME,
            Records=entries
        )

        failed_count = response.get("FailedPutCount", 0)
        if failed_count == 0:
            return len(records)

        falhas = [
            entries[i] for i, r in enumerate(response["RequestResponses"])
            if "ErrorCode" in r
        ]
        print(f"  [aviso] {failed_count} registros falharam, tentando de novo (tentativa {tentativa + 1})...")
        entries = falhas
        tentativa += 1
        time.sleep(2 ** tentativa)

    if entries:
        print(f"  [ERRO] {len(entries)} registros não foram enviados após {max_retries} tentativas.")
    return len(records) - len(entries)


def enviar_todos_em_lotes(df):
    eventos = gerar_todos_eventos(df)
    total = len(eventos)
    num_lotes = math.ceil(total / BATCH_SIZE)

    print(f"Enviando {total} eventos em {num_lotes} lote(s) de até {BATCH_SIZE}...")

    enviados = 0
    for i in range(0, total, BATCH_SIZE):
        lote = eventos[i:i + BATCH_SIZE]
        qtd_ok = enviar_lote(lote)
        enviados += qtd_ok
        print(f"Lote {i // BATCH_SIZE + 1}/{num_lotes}: {qtd_ok}/{len(lote)} enviados com sucesso.")

    print(f"\nTotal enviado: {enviados}/{total}")
    return enviados, total

In [12]:
enviados, total = enviar_todos_em_lotes(df_municipio)

Enviando 23995 eventos em 48 lote(s) de até 500...
Lote 1/48: 500/500 enviados com sucesso.
Lote 2/48: 500/500 enviados com sucesso.
Lote 3/48: 500/500 enviados com sucesso.
Lote 4/48: 500/500 enviados com sucesso.
Lote 5/48: 500/500 enviados com sucesso.
Lote 6/48: 500/500 enviados com sucesso.
Lote 7/48: 500/500 enviados com sucesso.
Lote 8/48: 500/500 enviados com sucesso.
Lote 9/48: 500/500 enviados com sucesso.
Lote 10/48: 500/500 enviados com sucesso.
Lote 11/48: 500/500 enviados com sucesso.
Lote 12/48: 500/500 enviados com sucesso.
Lote 13/48: 500/500 enviados com sucesso.
Lote 14/48: 500/500 enviados com sucesso.
Lote 15/48: 500/500 enviados com sucesso.
Lote 16/48: 500/500 enviados com sucesso.
Lote 17/48: 500/500 enviados com sucesso.
Lote 18/48: 500/500 enviados com sucesso.
Lote 19/48: 500/500 enviados com sucesso.
Lote 20/48: 500/500 enviados com sucesso.
Lote 21/48: 500/500 enviados com sucesso.
Lote 22/48: 500/500 enviados com sucesso.
Lote 23/48: 500/500 enviados com s

## Validação - lê de volta o que chegou em `bronze_stream/`

In [13]:
import s3fs

fs = s3fs.S3FileSystem()
fs.invalidate_cache()

arquivos = fs.glob(f"{BUCKET}/{PREFIX}**/*.parquet")
print(f"Arquivos encontrados: {len(arquivos)}")

dfs = [pd.read_parquet(arquivo, filesystem=fs) for arquivo in arquivos]
df_bronze_stream = pd.concat(dfs, ignore_index=True)

print(f"\nTotal de linhas lidas do bronze_stream: {len(df_bronze_stream):,}")
print(f"Total de eventos enviados (esperado): {total}")

print(f"\nLinhas duplicadas (por evento_id): {df_bronze_stream['evento_id'].duplicated().sum()}")
print(f"Valores nulos por coluna:")
print(df_bronze_stream.isna().sum())

print(f"\nSchema:")
print(df_bronze_stream.dtypes)

df_bronze_stream.head()

Arquivos encontrados: 1

Total de linhas lidas do bronze_stream: 23,995
Total de eventos enviados (esperado): 23995

Linhas duplicadas (por evento_id): 0
Valores nulos por coluna:
evento_id             0
tipo_evento           0
timestamp_evento      0
ano                   0
id_municipio          0
serie                 0
rede                  0
taxa_alfabetizacao    0
fonte                 0
dtype: int64

Schema:
evento_id              object
tipo_evento            object
timestamp_evento       object
ano                     int32
id_municipio           object
serie                   int32
rede                    int32
taxa_alfabetizacao    float64
fonte                  object
dtype: object


,evento_id,tipo_evento,timestamp_evento,ano,id_municipio,serie,rede,taxa_alfabetizacao,fonte
0,9973ebb2-34cd-477e-b5a5-e6093414dd0a,nova_medicao_indicador,2026-08-28T05:25:21.596256+00:00,2023,1100031,2,3,69.10,simulacao_streaming
1,48eb605a-a663-4eed-a4d9-f7c5f1e1a3bc,nova_medicao_indicador,2026-08-28T05:25:21.596347+00:00,2023,1100072,2,3,58.20,simulacao_streaming
2,19771192-0193-4997-8ecb-9447f846ca16,nova_medicao_indicador,2026-08-28T05:25:21.596389+00:00,2023,1100189,2,5,69.73,simulacao_streaming
3,5409d805-d466-49c3-b28e-cfd48fa22d53,nova_medicao_indicador,2026-08-28T05:25:21.596425+00:00,2023,1101609,2,3,50.70,simulacao_streaming
4,cd690422-05b8-42c6-8a98-7de3d3d5dec1,nova_medicao_indicador,2026-08-28T05:25:21.596461+00:00,2023,1101807,2,3,55.69,simulacao_streaming
